# Encuesta y ventas: calcula y cuadra

Parte A (celdas 1 a 4): una encuesta interna inventada. Parte B: ventas reales.

In [ ]:
# Solo en Colab o Jupyter: baja los CSV si no están en la carpeta.
import os, urllib.request

BASE = 'https://raw.githubusercontent.com/fjaltimiras/datacademy-labs/main/claude-code-workshop/07-datos-reportes/'
for archivo in ['encuesta.csv', 'ventas_reales.csv']:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(BASE + archivo, archivo)

## 1. Mirar

In [ ]:
import pandas as pd

encuesta = pd.read_csv('encuesta.csv')
print('Filas:', len(encuesta), '· notas vacías:', encuesta['nota'].isna().sum())
encuesta.head()

## 2. Promedio y cantidad por área

In [ ]:
areas = encuesta.groupby('area')['nota'].agg(notas='count', promedio='mean').sort_values('promedio', ascending=False)
areas.round(2)

## 3. Aplicar el mínimo y cuadrar

In [ ]:
MINIMO = 5
informe = areas.copy()
informe.loc[informe['notas'] < MINIMO, 'promedio'] = float('nan')
print('Usadas:', areas['notas'].sum(), '· excluidas:', encuesta['nota'].isna().sum(), '· total:', len(encuesta))
assert areas['notas'].sum() + encuesta['nota'].isna().sum() == len(encuesta)
informe.round(2)

## 4. Tu comprobación

¿Cuál sería el promedio general si, por error, los vacíos contaran como cero? Dos decimales.

In [ ]:
mi_respuesta = 0.0

if abs(mi_respuesta - round(encuesta['nota'].sum() / len(encuesta), 2)) < 0.005:
    print('Correcto. El promedio real es', round(encuesta['nota'].mean(), 2))
else:
    print('Todavía no coincide. Suma las notas y divide por todas las filas.')

## 5. Parte B: datos reales

Una semana de ventas (6 al 12 de junio de 2011) de una tienda en línea del Reino Unido. Montos en libras esterlinas.

Fuente: Chen, D. (2015). *Online Retail*. UCI Machine Learning Repository, [doi.org/10.24432/C5BW33](https://doi.org/10.24432/C5BW33), licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Cambios: se extrajo una semana, se tradujeron los nombres de las columnas y se quitó el identificador de cliente.

In [ ]:
ventas = pd.read_csv('ventas_reales.csv', dtype={'factura': str, 'codigo': str}, parse_dates=['fecha'])
ventas['monto'] = ventas['cantidad'] * ventas['precio']
print('Filas:', len(ventas), '· sin producto:', ventas['producto'].isna().sum(), '· precio cero:', (ventas['precio'] == 0).sum())
print('Cantidad negativa:', (ventas['cantidad'] < 0).sum(), '· filas idénticas a otra:', ventas.duplicated().sum())
ventas.head()

## 6. ¿Qué es una venta?

En esta fuente, una factura que empieza con C es una anulación. También hay ajustes con precio cero. Las filas idénticas no se borran: sin un número de línea no sabemos si son un error o dos unidades del mismo producto.

In [ ]:
es_anulacion = ventas['factura'].str.startswith('C')
bruto = ventas.loc[~es_anulacion, 'monto'].sum()
anulado = ventas.loc[es_anulacion, 'monto'].sum()
print(f'Bruto: {bruto:,.2f} · anulado: {anulado:,.2f} · neto: {bruto + anulado:,.2f}')
assert abs(bruto + anulado - ventas['monto'].sum()) < 0.01

## 7. El número que sorprende

Mira las dos filas de mayor monto, en positivo y en negativo. Están a tres minutos y son del mismo tamaño.

In [ ]:
raras = ventas.loc[ventas['monto'].abs().sort_values(ascending=False).index].head(2)
raras[['factura', 'producto', 'fecha', 'monto']]

Que una anule a la otra es una **hipótesis razonable**: coinciden el monto y la hora, pero la fuente no las une. El neto no cambia; el bruto y lo anulado, sí.

## 8. Por día

In [ ]:
import matplotlib.pyplot as plt

por_dia = pd.DataFrame({'bruto': ventas[~es_anulacion].groupby(ventas['fecha'].dt.date)['monto'].sum(), 'neto': ventas.groupby(ventas['fecha'].dt.date)['monto'].sum()})
por_dia.plot.bar(color=['#9aa4b8', '#00a383'], rot=0, title='Ventas por día (libras)')
plt.tight_layout()
plt.show()
por_dia.round(2)

## 9. Tu comprobación

El sábado no aparece. ¿Es venta cero o falta el dato? No lo sabemos: es una pregunta para quien lleva las ventas.

Ahora calcula: ¿qué porcentaje del **bruto** representa la factura de mayor monto? Un decimal.

In [ ]:
mi_porcentaje = 0.0

real = round(raras['monto'].max() / bruto * 100, 1)
if abs(mi_porcentaje - real) < 0.05:
    print('Correcto. Una sola fila pesa', real, '% del bruto de la semana.')
else:
    print('Todavía no coincide. Divide el monto mayor por el bruto y multiplica por 100.')